**NOTE** `phimats` environment should be used as kernel

In [1]:
import numpy as np

from PreProcessing import PhysicsConfig, MeshConfig, PreProcessing as PP

from MeshManager import MeshManager

from BoundaryConditions import *

from PostProcessing import WriteXDMF

%load_ext autoreload
%autoreload 2

### Simulation data

In [2]:
SimulName = "DoubleNotch"
# Element sets
nElementSets = 1
# Number of steps to achieve the load
nSteps = 300

### Read mesh file

In [ ]:
# Element name
elementName = "quad"  		# meshio compatible element name
mesh = MeshManager("DoubleNotch.msh", elementName)
mesh.WriteMesh("DoubleNotch")

In [4]:
# Create the config object
meshConfig = MeshConfig(
    nTotNodes=mesh.get_nTotNodes(),
    nTotElements=mesh.get_nTotElements(),
    nDim=mesh.get_nDim(),
    materialNames=mesh.getMaterialNames(),
)

### Apply mechanics boundary conditions
**NOTE** This is the total load to be achieved in `nSteps`.

In [7]:
dl = 0.6e-3   # [m]

In [ ]:
# Dirichlet BCs list
presBCs = []

presBCs.extend(AssignDirichletBC(mesh.getNodesByGroup("Left"), dof=1, value=dl))

presBCs.extend(AssignDirichletBC(mesh.getNodesByGroup("Top"), dof=1, value=dl))

presBCs.extend(AssignDirichletBC(mesh.getNodesByGroup("Left"), dof=0, value=0.0))

presBCs.extend(AssignDirichletBC(mesh.getNodesByGroup("Top"), dof=0, value=0.0))

presBCs.extend(AssignDirichletBC(mesh.getNodesByGroup("Right"), dof=0, value=0.0))

presBCs.extend(AssignDirichletBC(mesh.getNodesByGroup("Bottom"), dof=0, value=0.0))

presBCs.extend(AssignDirichletBC(mesh.getNodesByGroup("Right"), dof=1, value=0.0))

presBCs.extend(AssignDirichletBC(mesh.getNodesByGroup("Bottom"), dof=1, value=0.0))

# Write vtu for visualization 
WriteBCVTK(SimulName, mesh, presBCs, dofNames=['ux', 'uy'])

### Mechanical material data

In [9]:
# Analysis type
AnalysisType = "PlaneStrainPFF"
# Isotropy
Isotropy = "Isotropic"
# Young's modulus
Emod = 210e9
# Poisson's ratio
nu = 0.3

# Plasticity type
Plasticity = "IsoHard"
HardeningLaw = "PowerLaw"
sig_y0 = 350e6    # Yield strength (Pa)
K_hard = 1300e6    # Pa
n_pow = 0.1

rho_0 = 1e11  # Initial dislocation density (m⁻²)
M = 3 		  # Taylor factor
alpha = 0.3   # Dislocation interaction constant
b = 2.5e-10   # Burgers vector (m)
G = Emod/(2*(1+nu)) # Shear modulus
k1 = 2*(G/(200*G*alpha*b))  # Multiplication coefficient
k2 = 5  # Recovery coefficient

MechMaterial = {
	"DoubleNotch" : {
		"Elastic" : {
			"AnalysisType" : AnalysisType,
			"Isotropy" 	 : Isotropy,
			"Emod" 		 : Emod,
			"nu"   		 : nu,
		},
       	"Plastic" : {
			"Plasticity"   : Plasticity,
			"HardeningLaw" : HardeningLaw,
			"sig_y0"       : sig_y0,
			"K_hard"       : K_hard,
			"n_pow"        : n_pow,
			"rho_0"        : rho_0,
			"M"            : M,
			"b"            : b,
			"alpha"        : alpha,
			"k1"           : k1,
   			"k2"           : k2,
		},
	},
}

### PFF material data

In [ ]:
const_ell = 6e-5     # Length-scale regularization parameter [m]
wc = 90e6 # Critical work density [J/m³]

Gc = 2*const_ell*wc	 # Fracture toughness [J/m²]

PFFMaterial = {
	"DoubleNotch" : {
		"PFF" : {
			"const_ell" : const_ell,
			"wc" 	    : wc,
		}
    }
}

print("l: ", const_ell, " m")
print("Gc: ", Gc, " J/m²")
print("wc: ", wc, " J/m³")

### Initialize simulation object

### Mechanical

In [11]:
# Create the config object
mechPhysConfig = PhysicsConfig(
    SimulName = SimulName,
    PhysicsType = "Mechanical",
    PhysicsCategory = "Plastic",
    nSteps    = nSteps,
    presBCs=presBCs
)

In [ ]:
MechNotch = PP(mechPhysConfig, meshConfig, MechMaterial)

In [ ]:
MechNotch.WriteInputFile()

#### PFF

In [14]:
# Create the config object
pffPhysConfig = PhysicsConfig(
    SimulName = SimulName,
    PhysicsType = "PFF",
    PhysicsCategory = list(PFFMaterial["DoubleNotch"].keys())[0],
    nSteps    = nSteps,
)

In [ ]:
PFFNotch = PP(pffPhysConfig, meshConfig, PFFMaterial)

In [16]:
PFFNotch.WriteInputFile()

  Input file initialized: DoubleNotch.pff.in.hdf5


### Write output files

In [ ]:
OVERWRITE = False

In [ ]:
MechNotch.WriteOutputFile(overwrite=OVERWRITE)
PFFNotch.WriteOutputFile(overwrite=OVERWRITE)

### Write xdmf

In [ ]:
WriteXDMF(SimulName,    # Base simulation name
          "DoubleNotch",     # Mesh file 
          "quad",       # Element name
          nSteps+1,     # Number of steps 
          components=["mech", "pff"],  # Physics
    	  nDim=2,  # Dimensions
    	  mechModel="Plastic", # Adds strain_p, stress_eq, etc.
    	)

**PETSc command line options**

```
./DoubleNotch -snes_linesearch_type cp -snes_linesearch_damping 0.7 -snes_linesearch_monitor -snes_linesearch_max_it 50 -snes_max_it 400 -snes_max_funcs 10000
```